# Analysis — Portugal Rent Data, Q1 2026

This notebook explores the rent data I collected from INE to find the patterns
and numbers that ended up in the interactive map and article.

The main questions I want to answer:
1. How do rents vary across the country? Is the Lisbon-interior divide as stark as
   people say?
2. What does a median rent actually mean for someone on the minimum wage?
3. Which specific municipalities are the outliers — most and least expensive?
4. Is the Algarve actually cheaper than Lisbon, or is it just not talked about as much?

**Input file:** `portugal_rents.csv` (produced by `01_data_collection.ipynb`)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Keep plots clean and readable in the notebook
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False})

In [ ]:
df = pd.read_csv("portugal_rents.csv")
print(df.shape)
df.head()

## Quick overview

Let me separate the rows by level so I can look at them independently.
The `nuts2` rows are the big regions (Norte, Centro, Lisboa, Alentejo, Algarve);
the `nuts3` rows are the sub-regions; and `municipality` rows are individual
concelhos with enough lease data to get their own figure.

In [ ]:
nuts2  = df[df["level"] == "nuts2"]
nuts3  = df[df["level"] == "nuts3"]
munis  = df[df["level"] == "municipality"]

print(f"NUTS II regions:     {len(nuts2)}")
print(f"NUTS III sub-regions: {len(nuts3)}")
print(f"Municipalities:       {len(munis)}")

## 1. Regional picture (NUTS II)

Starting at the broadest level to get a sense of the north-south and
coastal-interior split.

In [ ]:
nuts2_sorted = nuts2.sort_values("price_m2", ascending=False)
nuts2_sorted[["zone", "price_m2", "rent_t1_45m2"]]

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(nuts2_sorted["zone"], nuts2_sorted["price_m2"], color="#d94f3d")
ax.set_xlabel("Median rent (€ / m²)")
ax.set_title("Median rent by NUTS II region, Q1 2026")
plt.tight_layout()
plt.show()

Interesting — Algarve is second highest, not far behind Lisboa. I'd expected the
beach region to be expensive but didn't realise it was *that* close to Lisbon
levels. The Norte region is almost 4 €/m² cheaper than Lisboa on average.

## 2. Which municipalities are the most expensive?

Now drilling down to individual municipalities.

In [ ]:
top10 = munis.sort_values("price_m2", ascending=False).head(10)
top10[["zone", "price_m2", "rent_t1_45m2"]]

In [ ]:
# And the cheapest
bottom10 = munis.sort_values("price_m2").head(10)
bottom10[["zone", "price_m2", "rent_t1_45m2"]]

In [ ]:
# Side-by-side comparison: most vs least expensive
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, data, title, color in [
    (axes[0], top10.sort_values("price_m2"),    "10 most expensive",  "#d94f3d"),
    (axes[1], bottom10.sort_values("price_m2", ascending=False), "10 cheapest", "#5b9bd5"),
]:
    ax.barh(data["zone"], data["price_m2"], color=color)
    ax.set_xlabel("€ / m²")
    ax.set_title(title)

plt.suptitle("Median rent by municipality, Q1 2026", y=1.02)
plt.tight_layout()
plt.show()

Lisboa at €17.42 is in a class of its own — about €3 more per m² than the
second most expensive municipality (Cascais). And the cheapest municipalities
are in the interior North/Centre, where €4-5 per m² is the norm.

The gap between the most expensive (Lisbon, €17.42) and cheapest municipality
in the dataset is massive. Let me calculate the ratio.

In [ ]:
cheapest = munis["price_m2"].min()
most_expensive = munis["price_m2"].max()
ratio = most_expensive / cheapest

print(f"Most expensive municipality: €{most_expensive:.2f} / m²")
print(f"Cheapest municipality:       €{cheapest:.2f} / m²")
print(f"Ratio:                        {ratio:.1f}×")

## 3. Affordability — what does this mean for someone on minimum wage?

The 2026 minimum wage in Portugal is **€920 / month** (gross). Let me calculate
what share of that goes to rent if you sign a new lease today, by municipality.

The usual rule of thumb is that housing should not exceed 30% of income.
Let me see how many municipalities are above that threshold for a minimum-wage
worker renting a 45 m² flat.

In [ ]:
MIN_WAGE = 920

munis = munis.copy()
munis["rent_burden_pct"] = (munis["rent_t1_45m2"] / MIN_WAGE * 100).round(1)

# How many municipalities cross the 30% threshold?
affordable = munis[munis["rent_burden_pct"] <= 30]
unaffordable = munis[munis["rent_burden_pct"] > 30]

print(f"Municipalities where T1 rent ≤ 30% of min wage: {len(affordable)}")
print(f"Municipalities where T1 rent >  30% of min wage: {len(unaffordable)}")

In [ ]:
# Which places are even remotely affordable on minimum wage?
munis.sort_values("rent_burden_pct")[["zone", "rent_t1_45m2", "rent_burden_pct"]]

In [ ]:
# Focus on Lisboa specifically
lisbon = munis[munis["zone"] == "Lisboa"].iloc[0]
print(f"Lisboa — rent: €{lisbon['rent_t1_45m2']:.0f} / month")
print(f"Lisboa — share of minimum wage: {lisbon['rent_burden_pct']:.1f}%")

That 85% figure is the one I led the article with — it's genuinely striking.
Even in cheaper municipalities, rent still eats up 30–40% of a minimum wage,
which is above the conventional affordability threshold.

## 4. Distribution of rents

Let me look at the spread more carefully. Is the distribution roughly normal,
or are there clear clusters?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

# Use all rows that have a meaningful price (nuts3 + municipalities)
plot_data = df[df["level"].isin(["nuts3", "municipality"])]["price_m2"]

ax.hist(plot_data, bins=10, color="#5b9bd5", edgecolor="white")
ax.axvline(plot_data.mean(), color="#d94f3d", linestyle="--", label=f"Mean €{plot_data.mean():.2f}")
ax.axvline(plot_data.median(), color="#333", linestyle="--", label=f"Median €{plot_data.median():.2f}")
ax.set_xlabel("Median rent (€ / m²)")
ax.set_ylabel("Count")
ax.set_title("Distribution of median rents (NUTS III + municipalities)")
ax.legend()
plt.tight_layout()
plt.show()

The distribution is right-skewed — most of the country sits in the 5–9 €/m²
range, but a cluster of high-value areas (Lisbon metro, Algarve) pulls the mean
above the median.

## 5. The Lisbon metro premium

Let me compare the municipalities inside and outside the Greater Lisbon
sub-region to see how much of an outlier the capital really is.

In [ ]:
# Grand Lisboa municipalities in the dataset
lisbon_metro = [
    "Lisboa", "Cascais", "Oeiras", "Sintra", "Amadora",
    "Loures", "Odivelas", "Vila Franca de Xira",
]
# Setúbal peninsula (often grouped with Lisbon in housing discussions)
setubal = ["Almada", "Seixal", "Setúbal"]

munis["area"] = munis["zone"].apply(
    lambda z: "Grande Lisboa" if z in lisbon_metro
    else ("Península de Setúbal" if z in setubal else "Resto do país")
)

munis.groupby("area")["price_m2"].agg(["mean", "min", "max"]).round(2)

In [ ]:
# National median for context
national = df[df["zone"] == "Portugal"]["price_m2"].values[0]
print(f"National median: €{national:.2f} / m²")

lisbon_mean = munis[munis["area"] == "Grande Lisboa"]["price_m2"].mean()
rest_mean = munis[munis["area"] == "Resto do país"]["price_m2"].mean()

premium = (lisbon_mean / rest_mean - 1) * 100
print(f"Grande Lisboa average is {premium:.0f}% above other municipalities")

## Key findings summary

Here are the numbers I ended up using in the article:

| Fact | Value |
|------|-------|
| National median rent | €9.46 / m² |
| Lisbon median rent | €17.42 / m² |
| Lisbon 1-bed (45 m²) | ~€784 / month |
| Share of min wage (€920) | ~85% |
| Most affordable municipality | ~€4.24 / m² (Terras de Trás-os-Montes) |
| Lisbon vs rest premium | ~60–70% above other municipalities |

The headline finding is that even outside Lisbon, new leases are barely
affordable for minimum-wage earners — the 30% rule is exceeded in most
municipality-level areas. The interior regions are significantly cheaper,
but the jobs are in the cities.

---
**What I'd do with more time:**
- Pull the quarterly time series to show how fast rents have risen over the last
  4–5 years (the % changes are dramatic).
- Compare rent-to-income ratios using municipal-level salary data, not just the
  national minimum wage.
- Look at the number of new leases per municipality to weight the analysis by
  market activity, not just price.